**Loading the Dataset**

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

# Set the path to the file you'd like to load
file_path = "Housing.csv"   # change if your file name is different

# Load the latest version
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "harishkumardatalab/housing-price-prediction",
    file_path,
)

print(df.head())
print(df.shape)
print(df.columns)

/tmp/ipykernel_1244/1804472584.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 29.3k/29.3k [00:00<00:00, 25.9MB/s]

      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  
(545, 13)
Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom

**Library Imports**

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

**Basic Cleaning**

In [3]:
df = df.copy()
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Expected target column
target = "price"

# Convert Yes/No columns to 1/0 if present
yes_no_cols = [
    "mainroad", "guestroom", "basement",
    "hot_water_heating", "airconditioning", "prefarea"
]

for col in yes_no_cols:
    if col in df.columns:
        df[col] = df[col].map({"yes": 1, "no": 0})

**Split features/target**

In [4]:
X = df.drop(columns=[target])
y = df[target]

# Identify column types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

**Preprocessing**

In [5]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

**Train-Test Splitting**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


**Linear Regression**

In [7]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression, k="all")),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
y_pred_lr = linear_model.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

**Random Forest Regressor**

In [8]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

**Results**

In [9]:
print("Linear Regression Results")
print(f"RMSE: {rmse_lr:.2f}")
print(f"R2  : {r2_lr:.4f}")
print()

print("Random Forest Results")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R2  : {r2_rf:.4f}")

Linear Regression Results
RMSE: 1324506.96
R2  : 0.6529

Random Forest Results
RMSE: 1393929.51
R2  : 0.6156
